
# Homework 4
## Interactions Between Land Surface and the Planetary Boundary Layer
Hourly data are provided from ERA5 reanalysis for the grid cell containing the DOE ARM field site in northern Oklahoma, site of the Land-Atmosphere Feedback Experiment (LAFE) field campaign during Summer 2017.<br>

Three NetCDF files are provided spanning the first 7 days of July 2017: 

* ea\_<strong>an\_ml</strong>\_lafe_hw3.nc: Model-level analyses (atmospheric state variables on model levels up to ~300hPa)

* ea\_<strong>an\_sfc</strong>\_lafe_hw3.nc: Single-level (horizontal surface -- no Z-coordinate) analyses (atmosphere and land state variables)

* ea\_<strong>fc\_sfc</strong>\_lafe_hw3.nc: Single-level forecast variables (fluxes and derived variables that are not defined instantaneously or are not prognostic in the ECMWF model)

\* <small> Wulfmeyer, V., and Coauthors, 2018: A new research approach for observing and characterizing land-atmosphere feedback. *Bull. Amer. Meteor. Soc.*, **99**, 1639-1667, https://doi.org/10.1175/BAMS-D-17-0009.1. </small>


In [ ]:
from datetime import datetime

import sys
import matplotlib.pyplot as plt
from matplotlib import cm

import numpy as np
import xarray as xr
import pandas as pd

# Here we will be using the MetPy library, which allows us to do some very meteorological things with the data
from metpy.plots import SkewT
from metpy.units import units
import metpy.calc as calc
import metpy.constants as c

import warnings
warnings.filterwarnings("ignore")

### About ERA5
ERA5 model levels are in a form of *sigma* coordinates in the vertical. In *sigma* coordinates, the mass of air in each layer (per unit horizontal area) maintains the same relative proportions in space and time. Typically they are on a scale from 0-1, where 1 means 100% of the mass of the atmosphere is above that level, 0 would be the "top" of the atmosphere. If surface pressure were 1000hPa, the 500hPa surface would be at $\sigma=0.5$. The reported values for model levels represent the value in the center of the layer, and the surface value is always 1.0 everywhere, regardless of elevation or actual surface pressure. 

In a *sigma* coordinate system, the actual pressure at any level is simply its *sigma* value times the surface pressure. 

In ERA5, this vertical coordinate is represented slightly differently, as pseudo-pressure levels. At each grid cell, the surface value is set equal to the mean sea level pressure value (101325 Pa) rather than 1.0. That means there is a little extra scaling to be done to find actual pressure: $p = \frac{\sigma_{ERA} \; p_{sfc }}{101325}$.

Below we read in a table of the values of this peculiar $\sigma_{ERA}$ coordinate from a .csv table - the values in the middle of each layer (`plevs`) and the top or interface with the level above (`ptops`) are provided.


In [ ]:
# MetPy does not define a constant for mean sea level pressure. We define it here.
mean_slp = 101325.0 * units.pascal

# This is a table of the pseudo-levels for ERA5 model-level data 
era5_levels = pd.read_csv('ERA5_levels.csv') 

# `plevs` is model pseudo-pressure in the middle of each layer,
# `ptops` is model pseudo-pressure at the top of each layer,
# `pbots` is model pseudo-pressure at the bottom of each layer.
# `ptops` of any layer equals `pbots` of the next layer above it.
plevs = era5_levels['plevel'].tolist() * units.pascals
###ptops = era5_levels['ptop'].tolist() * units.pascals
###pbots = ([mean_slp.m] + era5_levels['ptop'][:-1].tolist()) * units.pascals

# Surface and model-level data files (be sure they are in the same directory as this Notebook)
an_sfc_file = 'ea_an_sfc_lafe.nc'  # ERA5 analysis surface quantities (namely, state variables)
fc_sfc_file = 'ea_fc_sfc_lafe.nc'  # "forecast" quantities (namely fluxes)
an_ml_file = 'ea_an_ml_lafe.nc'    # multi-level state variables in troposphere (up to ~300 hPa)

### Extract Data

NetCDF data files; <strong>sfc</strong> files have only a time dimension that varies, although there are 1-D latitude and longitude dimensions to make them navigable by other software like GrADS.

The <strong>ml</strong> file has varying time and pseudo-pressure levels that need to be scaled by surface pressure (sp) to get real levels.

Read the code below, understand what each line is doing, and execute it.

In [ ]:
# Generate corresponding sigma levels for plevs and ptops
sigma_levs = plevs / mean_slp

# Define datasets for sounding (ml) and surface (sfc) data - read-only files
an_sfc_data = xr.open_dataset(an_sfc_file)
fc_sfc_data = xr.open_dataset(fc_sfc_file)
an_ml_data = xr.open_dataset(an_ml_file)

# Extract a time index for later use (it's the same for all 3 files)
all_times = an_ml_data['time']

# Standard sounding data -- we use MetPy's units support to assign units from NetCDF attributes to xarray DataArrays:
ml_t = an_ml_data['t'].squeeze().reindex(level=an_ml_data.level[::-1]) * units(an_ml_data['t'].attrs['units'])
ml_u = an_ml_data['u'].squeeze().reindex(level=an_ml_data.level[::-1]) * units(an_ml_data['u'].attrs['units'])
ml_v = an_ml_data['v'].squeeze().reindex(level=an_ml_data.level[::-1]) * units(an_ml_data['v'].attrs['units'])
ml_q = an_ml_data['q'].squeeze().reindex(level=an_ml_data.level[::-1]) * units(an_ml_data['q'].attrs['units'])
# `squeeze` removes the unitary horizontal dimensions, so we have only vertical and time.
# `reindex` flips the vertical profile so the 0th element is nearest the ground

# Also extract the surface pressure data (note - this is NOT sea level pressure - this site is well above sea level)
sfc_p = an_sfc_data['sp'].squeeze() * units(an_sfc_data['sp'].attrs['units'])

# It will be useful to have the pressures for all model levels and times in their own array:
ml_p, ml_sig = xr.broadcast(sfc_p,xr.DataArray(sigma_levs,coords={'level':ml_t['level']}))
ml_p = ml_p * ml_sig

ml_td = calc.dewpoint_from_specific_humidity(ml_p, ml_t, ml_q)
ml_w = calc.mixing_ratio_from_specific_humidity(ml_q)
ml_pot = calc.potential_temperature(ml_p, ml_t)

print(f"Typical array shape: {ml_t.shape}")

Note the shapes of the array examples above. 168 is the number of time steps (hours) in 7 days. 55 is the number of levels (the ECMWF atmospheric model has 137, but only the lowest 55 are used here). 


--------------------
## A. Mixing Diagrams

From *Santanello et al.* [2009], mixing diagrams are another way to plot boundary layer properties in a phase-space of thermodynamic variables. In its basic form, it is an hourly plot from sunrise to sunset (7AM through 6PM local time, or 12UTC through 23UTC) of the two terms comprising moist enthalpy: the latent energy of water vapor ($\lambda_v q$) on the X-axis, and the internal thermal energy of the air ($c_p T$) on the Y-axis. 

Commonly the 2m values of humidity and temperature are used, with the assumption that the daytime boundary layer is reasonably well mixed, so the near-surface values are representative of the evolution of boundary layer properties. 

Connecting adjacent time steps with lines traces the evolution of energy content of the boundary layer thoughout the day. Surface latent and sensible heat fluxes contribute to these changes. Averaging over the same daily period:

$$\lambda_v \Delta q_{sfc} = \frac {\overline{LE_{sfc}} \Delta t} {\overline{\rho_m} \overline{Z_{BL}}}, \quad
  c_p \Delta \theta_{sfc} = \frac {\overline{H_{sfc}} \Delta t} {\overline{\rho_m} \overline{Z_{BL}}}$$

**LE** and **H** are latent and sensible heat fluxes respectively.

\* <small>Santanello, J. A., C. D. Peters-Lidard, S. V. Kumar, C. Alonge, and W.-K. Tao, 2009: A Modeling and Observational Framework for Diagnosing Local Land–Atmosphere Coupling on Diurnal Time Scales. J. Hydrometeor., 10, 577–599, https://doi.org/10.1175/2009JHM1066.1.</small>

In [ ]:
# Let's extract the time series of near-surface temperature and dew point.
#   (yes, for the surface fields, ERA5 reports dew point, not specific humidity).
sfc_t = an_sfc_data['t2m'].squeeze() * units(an_sfc_data['t2m'].attrs['units'])
sfc_td = an_sfc_data['d2m'].squeeze() * units(an_sfc_data['d2m'].attrs['units'])

# We also need the PBL depth to estimate the surface heat flux contributions to the boundary layer.
#   Fortunately it is a diagnostic output in the reanalysis.
s_blh = an_sfc_data['blh'].squeeze() * units(an_sfc_data['blh'].attrs['units'])

# And the surface heat fluxes, which are in the forecast file, not the analysis file:
s_shf = -fc_sfc_data['msshf'].squeeze() * units(fc_sfc_data['msshf'].attrs['units']) 
s_lhf = -fc_sfc_data['mslhf'].squeeze() * units(fc_sfc_data['mslhf'].attrs['units'])  
# Note the minus signs. ECMWF declared positive downward for all surface fluxes, 
#     but the method of Santanello et al. (2009) assumes positive upward for these heat fluxes.

# Calculate surface potential temperature, profiles of mixing ratio and density
ml_w = calc.mixing_ratio_from_specific_humidity(ml_q)
ml_rho = calc.density(ml_p,ml_t,ml_w)
sfc_pot = calc.potential_temperature(sfc_p, sfc_t)

Let's create 1-dimensional arrays length 7, named `tot_le` and `tot_h` with appropriate units for the <u>daily</u> total daytime latent and sensible heating of the boundary layer:
$\lambda_v \Delta q_{sfc}$ and $c_p \Delta \theta_{sfc}$. We will include the daytime fluxes and states from 12UTC through 23UTC in the calculated totals for each day.

Also, you will have the boundary layer depth `s_blh` in meters, but otherwise your vertical coordinate for the model-layer variables is pressure. We will have to deal with that too - MetPy has functions to estimate this: `calc.add_height_to_pressure`.

In [ ]:
s_per_h = 3600.0 * units.second                # seconds per hour

# Create some empty arrays to fill with the daily flux totals (note the final units!)
tot_le = np.full_like(sfc_p[0:7].data.m, 0.0)  * units.joules / units.kg  
tot_h = np.full_like(sfc_p[0:7].data.m, 0.0)  * units.joules / units.kg

##############################################################
h0 = 12 # Starting hour (UTC)
h1 = 23 # Ending hour (UTC)

for d in np.arange(7):              # Loop through each day...
    t0 = datetime(2017, 7, d+1, h0) # YYYY, M, D, H(UTC) for start
    t1 = datetime(2017, 7, d+1, h1) # YYYY, M, D, H(UTC) for end

    ablh = s_blh.sel(time=slice(t0,t1)).mean() # average daytime BL height
    ablp = calc.add_height_to_pressure(sfc_p.sel(time=slice(t0,t1)).mean(), ablh) # average BL pressure level
    arho = calc.mixed_layer(ml_p.sel(time=slice(t0,t1)).mean(dim='time'), ml_rho.sel(time=slice(t0,t1)).mean(dim='time'),
                            depth=sfc_p.sel(time=slice(t0,t1)).mean()-ablp)      # average daytime BL density

    sum_le = s_lhf.sel(time=slice(t0,t1)).sum() * s_per_h                   # total daytime latent heat flux [J/m^2]
    sum_h = s_shf.sel(time=slice(t0,t1)).sum() * s_per_h                    # total daytime sensible heat flux [J/m^2]
#    print(d,": ",sum_le,s_per_h,"/",arho,ablh)
    tot_le[d] = sum_le.data / arho[0] / ablh.data      # calc.mixed_layer seems to have a bug (MetPy 1.1.0) - it returns a list
    tot_h[d] = sum_h.data / arho[0] / ablh.data        # That makes arho into a list of one number - weird!

# Plotting the Mixing Diagrams
x0, x1, y0, y1 = 22000, 59000, 292500, 319500
fig = plt.figure(figsize=(12, 9)) 
#sns.set_style("darkgrid", {"axes.facecolor": "#EAEAE2"})

##############################################################
for d in np.arange(7):              # Loop through each day...
    rgb = cm.get_cmap('tab10')(d)   # Good ol' "tab10" colors
    t0 = datetime(2017, 7, d+1, 12) # YYYY, M, D, H(UTC) for start
    t1 = datetime(2017, 7, d+1, 23) # YYYY, M, D, H(UTC) for end

    # Find the values of enthalpy to plot on the mixing diagram
    latent = c.Lv * calc.specific_humidity_from_dewpoint(sfc_p.sel(time=slice(t0,t1)), sfc_td.sel(time=slice(t0,t1)))
    sensible = c.Cp_d * calc.potential_temperature(sfc_p.sel(time=slice(t0,t1)), sfc_t.sel(time=slice(t0,t1)))
                                                             
    plt.plot(latent.data.m, sensible.data.m, color=rgb, marker="o", mfc='w', ms=6, label='Day '+str(d+1))
    plt.plot(latent[-1].data.m, sensible[-1].data.m, color=rgb, ms=8, lw=0, marker='D', mfc='w')
    plt.plot(latent[0].data.m, sensible[0].data.m, color=rgb, ms=8, lw=0, marker='D')

    # Surface heat fluxes:
    plt.arrow(latent[0].data.m, sensible[0].data.m, tot_le[d].m, tot_h[d].m,
              color=rgb, lw=0.5, head_width=400, length_includes_head=True)

    # Residuals (entrainment + advection + radiative cooling):
    plt.arrow(latent[0].data.m+tot_le[d].m, sensible[0].data.m+tot_h[d].m,
              latent[-1].data.m-(latent[0].data.m+tot_le[d].m),
              sensible[-1].data.m-(sensible[0].data.m+tot_h[d].m),
              color=rgb, lw=0.7, ls='--', head_width=300, length_includes_head=True)

# Constrain range of plot
plt.ylim(y0, y1) 
plt.xlim(x0, x1)

# Nice labeling 
plt.xlabel("Latent Energy [J/kg]", fontsize=14) 
plt.ylabel("Dry Enthalpy [J/kg]", fontsize=14)  
plt.title(f"Mixing Diagrams ({h0}-{h1} UTC)", fontsize=18)            
plt.legend(facecolor="#F6F6F6", loc='upper left', framealpha=1.0, fontsize=14)

# Manually-drawn supplemental legend for the arrows
box_props = dict(boxstyle='round', facecolor="#F6F6F6", edgecolor="#D9D9D9", alpha=1.0)
box_text = "Hourly Enthalpy (pluses and lines)\nSurface heat fluxes (thin arrows)\nResidual (dashed arrows)"
plt.text(x1-0.015*(x1-x0), y1-0.02*(y1-y0), box_text, fontsize=14, va='top', ha='right', bbox=box_props)

# Helpful extra labels along the axes
plt.text(x0+0.02*(x1-x0), y0-0.07*(y1-y0), "Dry", fontsize=14, fontweight='bold', va='bottom', ha='left', c='tab:brown')
plt.text(x1-0.02*(x1-x0), y0-0.07*(y1-y0), "Humid", fontsize=14, fontweight='bold', va='bottom', ha='right', c='c')
plt.text(x0-0.026*(x1-x0), y0+0.02*(y1-y0), "Cool", fontsize=14, fontweight='bold', va='bottom', ha='right', c='b')
plt.text(x0-0.026*(x1-x0), y1-0.02*(y1-y0), "Warm", fontsize=14, fontweight='bold', va='top', ha='right', c='r')

plt.grid(visible=True)
plt.show()


### Answer some questions

1. The slope of the surface heat flux vectors (solid lines with arrows) correspond to the daytime Bowen ratios. Describe why this is so. 
2. For each day, compare the "residual" vectors (dashed lines with arrows). The evening thermal component of the air $C_pT$ (y-axis) is always much larger than can be explained by surface sensible heat flux alone. Give at least three (3) other phenomena that could contribute to this difference in atmospheric thermal heat content.
3. The latent energy content of the air $\lambda_v q$ at the end of the day is much less than the surface latent heat flux contribution would imply, especially after the first couple of days. What could have happened to the evaporated moisture? Give two (2) possible reasons.
4. On the second day, the residual is quite small. Do you think the processes you listed to answer the first part of the question were not operating on these days? If they were, could something else be happening to cancel their effects? Discuss.
5. It rained on afternoon of the 2nd and evening of the 3rd, with the heaviest rains just after midnight (locally) on the 4th. After that there was no rain until a very small amount before sunset on the 7th. Could this change in the synotpic situations between the 3rd and 4th days have anything to do with features you discussed above? Explain.

----------------------
## Part B. The Heated Condensation Framework (HCF):

From *Tawfik et al.* [2015], the HCF is a method to determine the evolution of an atmospheric profile of temperature and humidity as it is heated from below. A number of thermodynamic variables are calculated based on the structure of the profile itself, rather than idealizations involving parcel theory. The technique can also be used to investigate the effect of moistening of the air at the surface by evaporation, and the combined effects of heating and moistening.

Heat is added to the atmospheric column by raising the surface temperature and mixing the air above it uniformly to the height where the profile potential temperature equals the surface potential temperature. The area on a skew-T log-P diagram between the temperature profile, the dry adiabat for the surface potential temperature and the surface pressure indicates the amount of heat added to the atmosphere. 

The mass of water vapor in the atmosphere  is also mixed to a uniform value: the mean of the specific humidity in that mixed layer. With enough added heat, the profiles of temperature and dew point will meet at the top of the mixed layer, where condensation would occur and a cloud would form. This level is the buoyant condensation level (BCL). However, in a dry atmosphere, it may require an unreasonable amount of thermal energy to trigger a cloud from sensible heating alone.

If some of the energy that warmed the air, represented as surface sensible heat flux, instead was used to evaporate water into the bottom of the atmopheric column (latent heat flux), the air would be more humid and the BCL would lower. As more energy is shifted from sensible to latent heat (the Bowen ratio approaches zero) the BCL would eventually drop to the ground. However, it may not do so monotonically. Inversions and dry layers in the atmosphere can drastically affect the evolution of the BCL calculation.

You can read Section 2 of *Tawfik et al.* [2015] to review the details of the procedure.

<ul> <li style="list-style-type: none;">
Tawfik, A. B., P. A. Dirmeyer, and J. A. Santanello, 2015: The Heated Condensation Framework. Part I: Description and Southern Great Plains Case Study. <i>J. Hydrometeorology</i>, <b>16</b>, 1929–1945, <a href="https://doi.org/10.1175/JHM-D-14-0117.1">doi: 10.1175/JHM-D-14-0117.1</a>.
</li> </ul>

The following function is provided to calculate a number of the HCF diagnostic variables from data at a specific time. Input fields to the function `find_bcl` are:
1. The evaporative fraction [float, scalar] associated with the calculation - determines the distribution of heat into the bottom of the atmosphere between latent and sensible. **This must be a non-negative number less than 1**.
2. The surface pressure [pint Quantity, scalar] - `sfc_p` from the code above, but only at one time step.
3. Profile of pressure on model levels [pint Quantity, vector] - `ml_p` at one time step.
4. Profile of temperature on model levels [pint Quantity, vector] - `ml_t` at one time step.
5. Profile of specific humidity on model levels [pint Quantity, vector] - `ml_q` at one time step.

Output is in the form of a class that will return 11 parameters, 9 pint Quantity scalars and two simple integers.
An example and description of output is provided below the following code cell.

In [ ]:
class result_container:
    pass
# This function expects all values at a single time
# ef is evaporative fraction (0 is all sensible heat, must be 0≤ef<1)
# sfc_p is surface pressure at single time
# ml_* are vertical profiles at a single time
def find_bcl(ef,sfc_p,ml_p,ml_t,ml_q,t_increment=0.05):
    """
    Calculates the Buoyant Condensation Level using Tawfik et al. (2015) formulation
    
    Required inputs:
        ef                  (float) = Evaporative fraction (must be in range 0 ≤ ef < 1)
        sf_p        (pint quantity) = Surface pressure 
        ml_p        (pint quantity) = Pressure profile
        ml_t        (pint quantity) = Temperature profile corresponding to ml_p
        ml_q        (pint quantity) = Specific humidity profile corresponding to ml_p

    Optional inputs:
        t_increment         (float) = Surface temperature stepping increment [K], to be multiplied by (1-ef) 
        
    Outputs:
        result              (class) = A class containing the following variables:
           heating  (pint quantity) = Total heat flux applied to reach condensation
           sensible (pint quantity) = Total sensible heat flux applied to reach condensation
           latent   (pint quantity) = Total latent heat flux applied to reach condensation
           moisture (pint quantity) = Total water added to atmosphere
           bclp     (pint quantity) = Buoyant condesation level pressure
           bclt     (pint quantity) = Buoyant condesation level temperature
           mixpot   (pint quantity) = Mixed layer potential temperature
           mixq     (pint quantity) = Mixed layer specific humidity
           bott     (pint quantity) = Bottom level temperature needed to reach BCL
           pmllev             (int) = Highest model level still below pml
           iter               (int) = Number of iterative steps to solve
    """
    
    # Some pre-checks
    if ml_t.shape != ml_p.shape:
        raise Exception("Sizes of temperature and pressure profiles do not match")
    if ml_q.shape != ml_p.shape:
        raise Exception("Sizes of humidity and pressure profiles do not match")
    if isinstance(sfc_p,xr.DataArray):
        sfc_p = sfc_p.data
    if isinstance(ml_p,xr.DataArray):
        ml_p = ml_p.data
    if isinstance(ml_t,xr.DataArray):
        ml_t = ml_t.data
    if isinstance(ml_q,xr.DataArray):
        ml_q = ml_q.data

    # Establish temperature increments at surface for iteration
    bot_p = ml_p[0]                                # lowest profile pressure level
    t_inc = (t_increment * (1-ef)) * units.kelvin  # step for increasing temperature
    t_start = ml_t[0]                              # Start at lowest profile T value
    bot_t = t_start
    h0_t = np.copy(ml_t)                           # Heat tally
    h0_q = np.copy(ml_q)                           # Moisture tally
    # Find mass per meter squared of lowest layer
    dep0 = 2.0 * (sfc_p - ml_p[0])
    mass0 = (dep0 / c.g).to_base_units()

    # Instead of just mixing humidity and working backwards to find BM,
    #   we add temperature & then moisture
    # Split that heat between SH and LH based on value of EF
    # Moisture is added to lowest model layer then mixed up thru mixed layer
                            
    try:  # Sometimes this has been known to fail - so using a "try" block
        iter_n = 0
        still_incr = True
        while still_incr:     # Iterative loop where we gradually inject heat into the atmosphere from below
            iter_n += 1
            bot_t += t_inc    # Nudge up the bottom temperature
            h0_t[0] = bot_t
            pml_pot = calc.potential_temperature(bot_p,bot_t) # Find the new dry adiabat
            #print(iter_n,bot_t)

            # Climb up the sounding and adjust T profile onto the pml_pot adiabat up to the top of PML
            for i in range(0, len(ml_p)):
                layer_t = calc.temperature_from_potential_temperature(ml_p[i], pml_pot)
                if (layer_t < ml_t[i]):
                    break
                h0_t[i] = layer_t # Neighborhood where we stopped

            # Values at the PML
            pml_int = (ml_t[i]-layer_t) / (ml_t[i]-layer_t+h0_t[i-1]-ml_t[i-1]) # Lin interp between layers
            pml_p = ml_p[i] + (ml_p[i-1]-ml_p[i]) * pml_int
            pml_t = h0_t[i] + (h0_t[i-1]-h0_t[i]) * pml_int
            depth_p = bot_p - pml_p
            pml_lev = i

            # Make special profiles for just the mixed layer with its top value at the PML
            mix_p = np.append(ml_p[:pml_lev],pml_p) 
            mix_t = np.append(ml_t[:pml_lev],pml_t) 
            mix_da = np.append(h0_t[:pml_lev],pml_t) # The dry adiabat for theta_BM
            
            # Total heat added in each layer is c_p/g*∆p*∆T
            # Sum up thru working mixed layer from sfc to PML
            mix_shf_tot = 0.0 * units.joule / units.m / units.m
            for i in range(0, len(mix_p)-1):
                delta_p = mix_p[i] - mix_p[i+1]
                delta_t = 0.5 * (mix_da[i] - mix_t[i] + mix_da[i+1] - mix_t[i+1])
                level_heat = c.Cp_d / c.g * delta_p * delta_t # J/m^2
                mix_shf_tot += level_heat

            # Latent heat, moisture needed to be added according to chosen EF
            mix_lhf_tot = mix_shf_tot * ef / (1 - ef)
            mix_moist_tot = (mix_lhf_tot / c.Lv).to_base_units()  # Water to add to atmosphere [kg/m^2]
            mix_q_add = (mix_moist_tot / mass0).to_base_units()   # Specific humidity boost [kg/kg]
            h0_q[0] = h0_q[0] + mix_q_add

            # Mix the moisture thru the mixed layer to the PML
            mix_q = calc.mixed_layer(ml_p,h0_q,bottom=bot_p,depth=depth_p)[0]

            # Find PML humidity deficit - opt out if we've reached it
            pml_qs = calc.specific_humidity_from_dewpoint(pml_p,pml_t)
            if mix_q >= pml_qs:
                still_incr = False
######## BCL Reached! ######################################

        # Adjust the BCL temperature & pressure based on specific humidity
        #   minimizes errors where profile is close to dry neutral
        mix_td = calc.dewpoint_from_specific_humidity(mix_p, layer_t, mix_q)
        mix_tc = mix_t.to('degC') # temperature profile from above

        # Find interval where Td crosses above T, lin interp final BCL level (P_BCL), T_BCL
        for i in range(len(mix_p)-1,0,-1):
            if (mix_td[i] > mix_tc[i] and mix_td[i-1] <= mix_tc[i-1]):
                pml_t = (mix_tc[i]+(mix_td[i]-mix_tc[i])*(mix_tc[i-1]-mix_tc[i])/(mix_td[i]-mix_tc[i]+mix_tc[i-1]-mix_td[i-1])).to('kelvin')
                pml_p = (mix_p[i]+(mix_td[i]-mix_tc[i])*(mix_p[i-1]-mix_p[i])/(mix_td[i]-mix_tc[i]+mix_tc[i-1]-mix_td[i-1]))
                break
        mix_flux_tot = mix_shf_tot + mix_lhf_tot

        result = result_container()
        # These output variables are pint Quantities having both magnitude and units:
        result.heating  = mix_flux_tot  # Total heat flux applied to reach condensation
        result.sensible = mix_shf_tot   # Total sensible heat flux applied to reach condensation
        result.latent   = mix_lhf_tot   # Total latent heat flux applied to reach condensation
        result.moisture = mix_moist_tot # Total water added to atmosphere
        result.bclp     = pml_p         # Buoyant condesation level pressure
        result.bclt     = pml_t         # Buoyant condesation level temperature
        result.mixpot   = pml_pot       # Mixed layer potential temperature
        result.mixq     = mix_q         # Mixed layer specific humidity
        result.bott     = bot_t         # Bottom level temperature needed to reach BCL
        # The following are integers
        result.pmllev   = pml_lev       # Highest model level still below pml
        result.iter     = iter_n        # Number of iterative steps to solve
    except:
        print(0,"Failure of function find_bcl")
#    return {'heating':heating,'sensible':sensible ,'latent':latent,'moisture':moisture,'bclp':bclp,'bclt':bclt,'mixpot':mixpot,'mixq':mixq,'bott':bott,'pmllev':pmllev,'itern':itern}
    return result
# Aren't you glad you didn't have to write this function yourself?!

As you may see from the embedded comments above, the function returns multiple quantities. For example, if the function is called as:

`out = find_bcl(ef,sfc_hcf_p,hcf_p,hcf_t,hcf_q)`

the results are returned as:

`out.heating`, `out.sensible` ,`out.latent` ,`out.moisture`,....

### Try it out

A functional example is given below (it may take a few seconds to run, depending on the number of iterations needed to converge).
* Run it for `ef = 0.0` (all sensible heat flux, no evaporation) and notice the results printed.
* Rerun with different values of `ef`: 0.1, 0.3, 0.5, etc.
* Try changing the time of day (Oklahoma is UTC-6, so hour 18 is noon, 06 is midnight) and look at different days (1 through 7 are valid choices).

In [ ]:
ef = 0.5
dt = datetime(2017, 7, 7, 15) # YYYY, M, D, H(UTC) for "sounding"
#tstep = np.where(all_times.data==np.datetime64(dt))[0][0]
sfc_hcf_p = sfc_p.sel(time=dt).data
hcf_p = ml_p.sel(time=dt).data
hcf_t = ml_t.sel(time=dt).data
hcf_q = ml_q.sel(time=dt).data

bcl = find_bcl(ef,sfc_hcf_p,hcf_p,hcf_t,hcf_q)

print(f"{bcl.iter} iterations for EF={ef:.2f} to determine the BCL is at: {bcl.bclp:.0f}s")
print(f"   This is {sfc_hcf_p-bcl.bclp:.0f}s or about {calc.add_pressure_to_height(2*units('m'),sfc_hcf_p-bcl.bclp).to('m'):.0f}s above the surface.")
print(f"   Bottom layer temperature was raised {(bcl.bott-hcf_t[0]):.1f} to: {bcl.bott:.1f}, {bcl.bott.to('celsius'):.1f}")
print(f"   Total surface sensible heat flux:  {bcl.sensible:.0f}")
print(f"   Total surface latent heat flux:    {bcl.latent:.0f}")
print(f"   Total heat flux:                   {bcl.heating:.0f}")
print(f"   Mixed layer potential temperature: {bcl.mixpot:.1f}")
print(f"   Mixed layer specific humidity:     {bcl.mixq:.3g}")

### B. Answer some questions

You can think of the BCL as the height where the base of the clouds would be if/when they form. 

Generally speaking, what relationship do you see between EF and...

1. ...the amount the temperature of the air has to rise to form a cloud?
2. ...the height of the cloud base above the ground?
3. ...the total heating energy (sensible+latent) needed to generate the cloud?
4. ...the humidity in the mixed layer once a cloud forms?

--------------
## Part C. HCF Continued:

For each day at 16UTC (about 10AM local time), the `find_bcl` function is run in a loop for a range of 9 values of evaporative fraction (EF): 0.1 to 0.9 in steps of 0.1. With the 7 days on the X axis, the following quantities are plotted by the cell below:
1. BCL pressure level in the atmosphere at each value of EF.
2. The EF and BCL each day that required the <u>least total heating</u> to trigger convection (i.e., the most efficient combination of sensible and latent heat flux to produce clouds).
3. The LCL each day estimated from only the surface meteorology (using parcel theory).
4. The maximum daytime PBL height (i.e., minimum pressure) during each day.

In [ ]:
local_hour = 10
edim = 9                  # number of EFs to cycle thru

utc_hour = local_hour + 6
if utc_hour > 23:
        raise ValueError("Can't go past 2300UTC with the local time")
del_ef = 0.8 / float(edim - 1)

bcl_lev = np.zeros((7,edim)) * units.pascal
tot_heat = np.zeros((7,edim)) * units(str(bcl.heating.units))
x_date = np.empty((7,), dtype=np.dtype('O'))
lcl_lev = np.zeros(7) * units.pascal
blh_lev = np.zeros(7) * units.pascal

warnings.filterwarnings('ignore',category=UserWarning) # Tends to throw needless warnings about supersaturation

# Generate data
print(f"Local hour: {local_hour:02d}")
for d in np.arange(7):
    t0 = datetime(2017, 7, d+1, 12) # YYYY, M, D, H(UTC) for start
    t1 = datetime(2017, 7, d+1, 23) # YYYY, M, D, H(UTC) for end
    dt = datetime(2017, 7, d+1, utc_hour) # YYYY, M, D, H(UTC) for "sounding"
    x_date[d] = dt
    #tstep = np.where(all_times.data==np.datetime64(dt))[0][0]
    sfc_hcf_p = sfc_p.sel(time=dt).data
    hcf_p = ml_p.sel(time=dt).data
    hcf_t = ml_t.sel(time=dt).data
    hcf_q = ml_q.sel(time=dt).data

    #blh_lev[d] = calc.add_height_to_pressure(sfc_hcf_p,np.max(s_blh[tstep-4:tstep+8]))
    blh_lev[d] = calc.add_height_to_pressure(sfc_hcf_p,s_blh.sel(time=slice(t0,t1)).max())
    lcl_lev[d],lclt = calc.lcl(sfc_p.sel(time=dt), sfc_t.sel(time=dt), sfc_td.sel(time=dt))
    print(f"  Day {dt}: LCL={lcl_lev[d]:.0f}, PBL height={blh_lev[d]:.0f} ",end="")
    for e in np.arange(edim):
        ef = float(e) * del_ef + 0.1
        bcl = find_bcl(ef,sfc_hcf_p,hcf_p,hcf_t,hcf_q)
        bcl_lev[d,e] = bcl.bclp
        tot_heat[d,e] = bcl.heating
        print(".",end="")
    print(" ")
print("### DONE ###")

In [ ]:
#Plot by EF value
plt.figure(figsize=(12,8))
ax = plt.axes()

# Y-axis is pressure
y0,y1 = 99000,75100
plt.ylim(y0,y1)
plt.xlim(all_times[0],all_times[-1])
cmap = cm.get_cmap("turbo_r")

# Plot the heights where clouds would form for each EF
for e in np.arange(edim): # Loop through the EFs, each across all 7 days
    ef = float(e)*0.1 + 0.1   
    rgb = cmap(ef) 
    plt.plot(x_date,bcl_lev[:,e],color=rgb,marker="o") # A line for each EF, X-axis is each day

# Plot other boundary layer information about each day
lowest = np.argmin(tot_heat,axis=1) # Index of lowest total energy each day
low_ef = lowest*0.1 + 0.1           # The EF value for lowest total energy
for d in np.arange(7):    # Loop through days to find the other statistics
    plt.plot(x_date[d],bcl_lev[d,lowest[d]],ls='None',color='k',marker="o",ms=10,fillstyle='none') # Lowest total energy 
    plt.plot(x_date[d],lcl_lev[d],ls='None',color='k',marker="_",ms=50,mew=2)            # LCL
    plt.plot(x_date[d],blh_lev[d],ls='None',color='g',marker="D",ms=8)                   # PBL depth
    plt.text(x_date[d],y0-1000,f'{low_ef[d]:.1}',c='k',ha='center',va='top',fontsize=14) # EF value in black for lowest energy

plt.xticks(rotation=60)
plt.ylabel('BCL level [Pa]',fontsize=14)
plt.legend(['EF=0.1','0.2','0.3','0.4','0.5','0.6','0.7','0.8','0.9'],bbox_to_anchor=(1.02,1.00),loc='upper left',borderaxespad=0.)

plt.text(1.02,0.52,"◆ Max Daytime\n    PBL Height",transform=ax.transAxes,fontsize=12,color='g',va='bottom',ha='left')
plt.text(1.02,0.46,"— LCL",transform=ax.transAxes,fontsize=12,color='k',va='center',ha='left')
plt.text(1.02,0.056,"⊙ Least Energy\n    to Reach BCL",transform=ax.transAxes,fontsize=12,color='k',va='top',ha='left')

plt.grid(visible=True)

plt.show()

### C: Answer some more questions

1. There is a large amount of variability from day to day in the sensitivity of cloud formation and the cloud base (BCL) depending on the partitioning of surface fluxes between latent and sensible heat. The days early in the period have little spread, especially if we neglect EF=0.1, while there is more spread across values of EF late in the period, especially on the 6th. What does this suggest regarding the sensitivity of the atmosphere to different land surface states and properties across these seven days?
2. The most efficient choice of EF to produce clouds is not constant, but varies from day to day (and in fact hour by hour, as you will see if you repeat the plot for different times). Why is this so? What determines this response?
3. The base of the clouds estimated from the LCL does not agree very well with the results from the more complex HCF approach. Discuss why this is so - there are at least two (2) distinct reasons you can invoke.
4. The maximum PBL height, reached each afternoon, is much higher than any of the LCL values or nearly all of the BCL estimates. Why? You may want to recrate the plot for other hours of the day, and draw on what you (hopefully) learned in Part A to help answer this question.
5. Daytime warming of the land surface heats the PBL, and entrainment of higher potential temperature air from above heats the PBL. Yet we don't see the lower atmosphere just continue to get hotter and hotter day after day. What offsets all this heating (and when)? 